In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    log_loss,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix
)


device = torch.device("mps")

# load BERT
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def extract_all_text_features(data_series, batch_size=32):
    print("Starting BERT extraction (this only runs if no cached file is found)...")
    data = [str(text) if pd.notna(text) else "" for text in data_series]
    full_list = []
    
    # process in batches
    for i in range(0, len(data), batch_size):
        if i % 1000 == 0:
            print(f"Processing row {i} / {len(data)}")
            
        batch = data[i: i+batch_size]
        text_input = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        text_input = {k: v.to(device) for k, v in text_input.items()}
        
        with torch.no_grad():
            outputs = bert_model(**text_input)
            
        CLS_token = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        full_list.append(CLS_token)
        
    return np.vstack(full_list)

def apply_modality_dropout(text_feats, tab_feats, p_text, p_tabular):
    """Zeroes out entire modalities for a given percentage of training samples."""
    n_samples = text_feats.shape[0]
    
    # create boolean masks (True means we KEEP the modality)
    keep_text_mask = np.random.rand(n_samples) >= p_text
    keep_tab_mask = np.random.rand(n_samples) >= p_tabular
    
    # apply masks: if mask is False, the row becomes all zeros
    text_dropped = text_feats.copy()
    text_dropped[~keep_text_mask] = 0.0
    
    tab_dropped = tab_feats.copy()
    tab_dropped[~keep_tab_mask] = 0.0
    
    return text_dropped, tab_dropped

def evaluate_metrics(y_true, y_pred, y_prob, label):
    """Calculates and prints core model evaluation metrics."""
    print(f"\n--- {label} ---")
    ll = log_loss(y_true, y_prob)
    print("Log Loss (Error):", f"{ll:.4f}")

    print("Accuracy:", f"{accuracy_score(y_true, y_pred):.3f}")
    print("Balanced Accuracy:", f"{balanced_accuracy_score(y_true, y_pred):.3f}")
    print("Macro F1:", f"{f1_score(y_true, y_pred, average='macro'):.3f}")
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    print("Cohen's Kappa (Quadratic):", f"{kappa:.3f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

In [ ]:
### ROMAL YOU'RE GONNA HAVE TO CHANGE THESE FILE PATHS ###
file_path = "/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-alladults.csv"
processed_data = pd.read_csv(file_path)

cache_path = "/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/bert_embeddings_cache.npy"

# check if we already did the heavy lifting
if os.path.exists(cache_path):
    print(f"Found cached BERT embeddings at {cache_path}. Loading into memory...")
    all_text_features = np.load(cache_path)
else:
    print("No cache found. Extracting BERT embeddings...")
    all_text_features = extract_all_text_features(processed_data['chiefcomplaint'])
    np.save(cache_path, all_text_features)
    print(f"Saved embeddings to {cache_path} for future use.")

processed_data['y'] = processed_data['acuity'] - 1
y_vals = processed_data['y'].values

indices = np.arange(len(processed_data))
idx_train, idx_test, y_train, y_test = train_test_split(
    indices, y_vals, test_size=0.2, random_state=0, stratify=y_vals
)

train_text = all_text_features[idx_train]
test_text = all_text_features[idx_test]

cols_to_drop = ['chiefcomplaint', 'race', 'acuity', 'y']
tabular_data = processed_data.drop(columns=cols_to_drop)

x_train_tabular = tabular_data.iloc[idx_train].copy()
x_test_tabular = tabular_data.iloc[idx_test].copy()

train_mean = x_train_tabular.mean()
x_train_tabular = x_train_tabular.fillna(train_mean)
x_test_tabular = x_test_tabular.fillna(train_mean)

scaler = StandardScaler()
train_tabular = scaler.fit_transform(x_train_tabular)
test_tabular = scaler.transform(x_test_tabular)

combined_test = np.hstack((test_text, test_tabular))

Found cached BERT embeddings at /Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/bert_embeddings_cache.npy. Loading into memory...


In [ ]:
def run_experiments(dropout_pairs):
    """Runs training loops over a given set of dropout configurations."""
    save_dir = "/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/early_fusion_models"
    os.makedirs(save_dir, exist_ok=True)
    
    for p_text, p_tabular in dropout_pairs:
        print(f"\n==================================================")
        print(f"Training with Dropout - Text: {p_text*100}%, Tabular: {p_tabular*100}%")
        print(f"==================================================")
        
        # apply dropout
        train_text_dropped, train_tabular_dropped = apply_modality_dropout(
            train_text, train_tabular, p_text, p_tabular
        )
        
        # fuse modalities
        combined_train = np.hstack((train_text_dropped, train_tabular_dropped))
        
        fusion_classifier = MLPClassifier(hidden_layer_sizes=(256, 256), max_iter=5000, random_state=42)
        fusion_classifier.fit(combined_train, y_train)

        # save model state
        model_path = os.path.join(save_dir, f"earlyfusion_text{int(p_text*100)}_tab{int(p_tabular*100)}.joblib")
        joblib.dump({
            "model": fusion_classifier,
            "scaler": scaler,
            "train_mean": train_mean,
            "tabular_columns": x_train_tabular.columns.tolist()
        }, model_path)
        
        # make predictions
        predictions = fusion_classifier.predict(combined_test)
        probabilities = fusion_classifier.predict_proba(combined_test) # needed for log_loss
        
        evaluate_metrics(y_test, predictions, probabilities, 
            label=f"Evaluation (Text Drop: {p_text*100}%, Tab Drop: {p_tabular*100}%)")

In [7]:
no_dropout = [
    (0.00, 0.00)
]
run_experiments(no_dropout)


Training with Dropout - Text: 0.0%, Tabular: 0.0%

--- Evaluation (Text Drop: 0.0%, Tab Drop: 0.0%) ---
Log Loss (Error): 1.5274
Accuracy: 0.566
Balanced Accuracy: 0.347
Macro F1: 0.363
Cohen's Kappa (Quadratic): 0.366

Confusion Matrix:
[[ 2106  1348  1309    42     4]
 [ 1141  9732 17399   347    34]
 [  570  8391 37786  1102    85]
 [   50   753  5576   860    85]
 [    7    65   240   108    24]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.44      0.49      4809
           1       0.48      0.34      0.40     28653
           2       0.61      0.79      0.69     47934
           3       0.35      0.12      0.18      7324
           4       0.10      0.05      0.07       444

    accuracy                           0.57     89164
   macro avg       0.42      0.35      0.36     89164
weighted avg       0.54      0.57      0.54     89164

